# 05. Accessibilité de la population aux transports structurants
## Métropole Rouen Normandie — Situation 2026 et réseau cible SERM

## §0 — Objectif et cadrage

Ce notebook produit le **livrable analytique principal** du projet : pour chaque
carreau de population Filosofi 200 m, le meilleur niveau de desserte structurante
atteignable à pied ou à vélo, pour les deux horizons (situation actuelle 2026 et
réseau cible SERM), puis — par agrégation pondérée par la population — le profil
d'accessibilité de la Métropole Rouen Normandie.

### Distinction fondamentale vis-à-vis de nb03

`03_isochrones.ipynb` calcule l'**atteignabilité** : depuis chaque arrêt,
quels nœuds du réseau viaire sont accessibles en moins de 15 minutes ?
Ce notebook calcule l'**accessibilité** : depuis chaque lieu d'habitation,
quel est le meilleur niveau de desserte structurante atteignable ?

Le passage de l'un à l'autre repose sur l'inversion du sens de lecture :
les carreaux de population sont rattachés au réseau viaire (snap-to-edge,
§3), et le meilleur niveau est lu via la table `acces_noeuds` de nb03 (§4).

---

### Hiérarchie des niveaux de desserte

`aucun < Bus < TEOR < Métro < Car express < Train`

Le bac est exclu (structurant sur un seul corridor, marginal en population).
Le car express n'apparaît qu'à l'horizon SERM ; son absence de la situation
2026 est traitée comme un niveau non atteint (`aucun` ou niveau inférieur).

---

### Prérequis — notebooks à exécuter au préalable

| Notebook | Sortie requise |
|---|---|
| `01_acquisition_donnees_OSM.ipynb` | `data/reseau_pieton_MRN.graphml`, `data/reseau_velo_MRN.graphml` |
| `03_isochrones.ipynb` (HORIZON = "2026") | `data/acces_noeuds_2026.gpkg` |
| `03_isochrones.ipynb` (HORIZON = "SERM") | `data/acces_noeuds_SERM.gpkg` |
| `04_logements.ipynb` | `data/population_carreaux_MRN.gpkg` |

---

### Entrées

| Fichier | Description |
|---|---|
| `data/reseau_pieton_MRN.graphml` | Graphe OSMnx piéton, EPSG:2154 |
| `data/reseau_velo_MRN.graphml` | Graphe OSMnx vélo, EPSG:2154 |
| `data/acces_noeuds_2026.gpkg` | Table nœud × niveau × temps d'accès, horizon 2026 |
| `data/acces_noeuds_SERM.gpkg` | Table nœud × niveau × temps d'accès, réseau cible SERM |
| `data/population_carreaux_MRN.gpkg` | Carreaux Filosofi 200 m, population par carreau, MRN stricte |

**Contrat d'interface nb03 → nb05** (`acces_noeuds_{horizon}.gpkg`) :

| Colonne | Type | Détail |
|---|---|---|
| `mode_deplacement` | str | `"piéton"` \| `"vélo"` |
| `node` | int64 | osmid du nœud |
| `niveau_ordre` | int64 | rang hiérarchique (1 Bus … 5 Train) |
| `niveau` | str | libellé (`"Bus"` … `"Train"`) |
| `temps_acces_s` | int64 | temps d'accès minimal arrêt → nœud, **en secondes** |
| `tranche_min` | int64 | tranche cartographique 5 / 10 / 15, dérivée de `temps_acces_s` |
| `horizon` | str | `"2026"` \| `"SERM"` |
| `geometry` | Point | EPSG:2154 |

> Le `temps_acces_s` est la colonne porteuse du contrat : il permet de retrancher
> le tronçon d'approche du budget-temps en §4, ce qu'une simple tranche catégorielle
> rendrait impossible.

---

### Sortie

| Fichier | Description |
|---|---|
| `data/population_accessibilite_MRN.gpkg` | Un enregistrement par carreau × mode de déplacement. Colonnes analytiques principales : `niveau_actuel`, `tranche_actuel`, `niveau_cible`, `tranche_cible` (meilleur niveau atteignable par horizon), `pop_men` (population du carreau). Sert de base à nb06 (comparaison avant / après) et aux cartes de restitution. |

---

### Structure du notebook

| § | Titre | Rôle |
|---|---|---|
| §1 | Configuration | `trouver_racine()`, imports, constantes, CRS |
| §2 | Chargement et contrôles d'intégrité | Carreaux, tables `acces_noeuds`, graphes ; vérification du contrat d'interface |
| §3 | Snap-to-edge : rattachement des carreaux au réseau | `representative_point()`, `ox.distance.nearest_edges`, projection sur l'arête |
| §4 | Affectation du niveau atteignable par carreau | Lecture via les deux nœuds extrémités, correction du tronçon d'approche sur le budget-temps |
| §5 | Indicateur composite d'accessibilité | Meilleur niveau × tranche × population ; c'est ici que naît l'accessibilité au sens du cadrage |
| §6 | Validation croisée | Point-dans-polygone contre `isochrones_2026`, réconciliation des totaux de population |
| §7 | Export et limites | Écriture de `population_accessibilite_MRN.gpkg`, entrée Limites README |

## §1 — Configuration

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd

print(f"OSMnx version  : {ox.__version__}")
print(f"GeoPandas version : {gpd.__version__}")

# Système de référence unique du projet
CRS_PROJET = "EPSG:2154"

# Seuil d'accessibilité et gradient (doivent coïncider avec nb03)
SEUIL_ACCESSIBILITE_MIN = 15        # minutes
SEUIL_ACCESSIBILITE_S   = SEUIL_ACCESSIBILITE_MIN * 60  # secondes (unité de acces_noeuds)
TRANCHES_MIN = [5, 10, 15]          # gradient cartographique

Détection de la racine du projet via le fichier sentinelle `.projectroot`,
puis ancrage de tous les chemins dessus. Cette approche est indépendante du
répertoire de travail : elle fonctionne sous JupyterLab comme sous VSCode,
quel que soit le réglage `notebookFileRoot`, et après un clone comme après un
téléchargement ZIP du dépôt.

In [ ]:
def trouver_racine(marqueur=".projectroot"):
    """Remonte depuis le cwd jusqu'au dossier contenant `marqueur`."""
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if (parent / marqueur).exists():
            return parent
    raise FileNotFoundError(f"Racine introuvable (marqueur {marqueur!r})")


ROOT    = trouver_racine()
DATA_DIR = ROOT / "data"
print(f"Racine du projet : {ROOT}")

Chargement des graphes viaires produits par `01_acquisition_donnees_OSM.ipynb`.
Ils servent ici à deux usages distincts des graphes de nb03 : localiser l'arête
la plus proche de chaque carreau (`ox.distance.nearest_edges`) et lire la
géométrie des nœuds extrémités pour la correction du tronçon d'approche (§4).
Le filtrage `largest_component` de nb03 n'est pas répété : il n'est nécessaire
que pour le routage depuis les arrêts ; `nearest_edges` opère sur la géométrie
des arêtes et reste cohérent sur le graphe complet.

In [ ]:
G_pieton = ox.load_graphml(DATA_DIR / "reseau_pieton_MRN.graphml")
G_velo   = ox.load_graphml(DATA_DIR / "reseau_velo_MRN.graphml")

for nom, G in {"piéton": G_pieton, "vélo": G_velo}.items():
    crs = G.graph.get("crs", "non renseigné")
    print(
        f"Réseau {nom:<7} | "
        f"{G.number_of_nodes():>7,} nœuds | "
        f"{G.number_of_edges():>7,} arêtes | "
        f"CRS : {crs}"
    )

GRAPHES = {"piéton": G_pieton, "vélo": G_velo}

## §2 — Chargement et contrôles d'intégrité

Chargement des carreaux de population produits par `04_logements.ipynb`.
La colonne analytique est `ind_pond` : population Filosofi pondérée au prorata
de la surface conservée après découpe au périmètre MRN (carreaux de bordure).
`ind` porte la population brute du carreau INSEE, utile pour la réconciliation.

In [ ]:
carreaux = gpd.read_file(DATA_DIR / "population_carreaux_MRN.gpkg")

print(f"Carreaux chargés : {len(carreaux):,}")
print(f"CRS              : {carreaux.crs}")
print(f"Population totale (ind_pond) : {carreaux['ind_pond'].sum():,.0f}")
print(f"Colonnes : {list(carreaux.columns)}")

Contrôles d'intégrité des carreaux : CRS, colonnes attendues, population
positive, absence de géométries nulles.

In [ ]:
COLS_CARREAUX = {"idcar_200m", "ind", "ind_pond", "frac", "geometry"}
manquantes = COLS_CARREAUX - set(carreaux.columns)
assert not manquantes, f"Colonnes manquantes dans population_carreaux_MRN : {manquantes}"
assert carreaux.crs is not None and carreaux.crs.to_epsg() == 2154, (
    f"CRS inattendu : {carreaux.crs}"
)
assert carreaux.geometry.notna().all(), "Géométrie(s) nulle(s) dans les carreaux"
assert (carreaux["ind_pond"] >= 0).all(), "Population pondérée négative"

POP_TOTALE = carreaux["ind_pond"].sum()
print(f"Contrôles carreaux OK — {len(carreaux):,} carreaux | pop totale : {POP_TOTALE:,.0f}")

Chargement des tables d'atteignabilité produites par `03_isochrones.ipynb`
pour les deux horizons. Le contrat d'interface est vérifié immédiatement après
le chargement de chaque table (cf. §0 — colonnes, types, unités, bornes).

In [ ]:
def controler_acces_noeuds(gdf, horizon):
    """Fige le contrat nb03 → nb05.
    Toute dérive de schéma fait échouer nb05 ici, plutôt que de produire
    un résultat faux silencieusement."""
    COLS_ATTENDUES = {
        "mode_deplacement", "node", "niveau_ordre", "niveau",
        "temps_acces_s", "tranche_min", "horizon", "geometry",
    }
    manquantes = COLS_ATTENDUES - set(gdf.columns)
    assert not manquantes, (
        f"acces_noeuds_{horizon} : colonnes manquantes {manquantes}"
    )
    assert gdf.crs is not None and gdf.crs.to_epsg() == 2154, (
        f"acces_noeuds_{horizon} : CRS attendu EPSG:2154, trouvé {gdf.crs}"
    )
    assert set(gdf["mode_deplacement"]) <= {"piéton", "vélo"}, (
        f"acces_noeuds_{horizon} : mode_deplacement inattendu "
        f"{set(gdf['mode_deplacement']) - {'piéton', 'vélo'}}"
    )
    assert gdf["temps_acces_s"].between(0, SEUIL_ACCESSIBILITE_S).all(), (
        f"acces_noeuds_{horizon} : temps_acces_s hors [0, {SEUIL_ACCESSIBILITE_S}] s"
    )
    assert (gdf["temps_acces_s"] <= gdf["tranche_min"] * 60).all(), (
        f"acces_noeuds_{horizon} : temps_acces_s incohérent avec tranche_min"
    )
    assert (gdf["horizon"] == horizon).all(), (
        f"acces_noeuds_{horizon} : étiquette horizon ≠ '{horizon}'"
    )
    print(
        f"acces_noeuds_{horizon} : {len(gdf):,} lignes | "
        f"{gdf['node'].nunique():,} nœuds distincts | schéma conforme"
    )


acces_noeuds = {
    h: gpd.read_file(DATA_DIR / f"acces_noeuds_{h}.gpkg")
    for h in ("2026", "SERM")
}
for h, gdf in acces_noeuds.items():
    controler_acces_noeuds(gdf, h)

Réconciliation des totaux de population avec nb04. L'écart structurel (~8 %)
entre population Filosofi (`ind_pond`) et population municipale est documenté
dans les Limites du README ; on vérifie ici que le chiffre n'a pas dérivé
au-delà de la fourchette attendue — signe d'une erreur de traitement.

In [ ]:
POP_MUNICIPALE_REF = 491_000   # population municipale MRN 2021 (INSEE)
ECART_MAX = 0.12               # tolérance : jusqu'à 12 % d'écart accepté

ecart = abs(POP_TOTALE - POP_MUNICIPALE_REF) / POP_MUNICIPALE_REF
print(f"Population Filosofi pondérée : {POP_TOTALE:>10,.0f}")
print(f"Population municipale 2021   : {POP_MUNICIPALE_REF:>10,}")
print(f"Écart relatif                : {ecart:>10.1%}  (tolérance ≤ {ECART_MAX:.0%})")

assert ecart <= ECART_MAX, (
    f"Écart de population hors tolérance : {ecart:.1%} > {ECART_MAX:.0%} — "
    "vérifier le découpage ou la source Filosofi"
)
print("Réconciliation OK — écart dans la fourchette documentée (~8 %).")

## §3 — Snap-to-edge : rattachement des carreaux au réseau

Le §4 lit le meilleur niveau de desserte atteignable depuis chaque carreau via
la table `acces_noeuds`. Pour cela il faut d'abord localiser chaque carreau sur
le réseau viaire — c'est le snap.

**Pourquoi snap-to-edge et non snap-to-node ?**
En nb03 le snap-to-node est défendable pour les *arrêts* : les modes
structurants sont situés dans des zones bien maillées, et le tronçon résiduel
(médiane < 25 m) est négligeable devant le budget de 15 min.
Pour les *carreaux*, la situation est inverse : les 5 915 carreaux couvrent
l'ensemble de la MRN, y compris les zones rurales traversées par de longues
arêtes OSM non subdivisées. Un snap-to-node placerait le centroïde d'un carreau
rural à mi-chemin d'une arête de 800 m sur un nœud extrémité, ajoutant jusqu'à
400 m d'erreur. Le snap-to-edge (projection orthogonale sur l'arête la plus
proche) ramène cette erreur à zéro le long de l'arête.

**Point représentatif plutôt que centroïde.**
Les carreaux tronqués en limite de MRN ont un centroïde qui peut tomber hors
de leur surface découpée. `representative_point()` garantit un point intérieur
dans tous les cas — condition nécessaire pour que `nearest_edges` ne projette
pas un carreau de bordure sur une arête du département voisin.

In [ ]:
# Point représentatif de chaque carreau (intérieur garanti, même pour les
# carreaux tronqués en limite de MRN)
carreaux["pt_repr"] = carreaux.geometry.representative_point()

X = carreaux["pt_repr"].x.to_numpy()
Y = carreaux["pt_repr"].y.to_numpy()

print(f"Points représentatifs calculés : {len(carreaux):,}")
print(f"NaN X : {sum(1 for x in X if x != x)} | NaN Y : {sum(1 for y in Y if y != y)}")

`ox.distance.nearest_edges` retourne, pour chaque point, le triplet
`(u, v, key)` identifiant l'arête la plus proche et la distance orthogonale
jusqu'à elle. Cette distance est la longueur du tronçon d'approche — elle sera
convertie en secondes et retranchée du budget-temps en §4.

Le snap est calculé une fois par réseau (piéton, vélo) : les deux graphes
ont des topologies légèrement différentes (OSMnx filtre les tronçons
non empruntables à pied ou à vélo) et peuvent retourner des arêtes distinctes
pour le même carreau.

In [ ]:
SUFFIXE = {"piéton": "pieton", "vélo": "velo"}

for nom, G in GRAPHES.items():
    suf = SUFFIXE[nom]
    # nearest_edges retourne (edges, dists) où edges est un tableau de triplets
    # (u, v, key) — décompression en deux temps
    edges, dist_arr = ox.distance.nearest_edges(G, X, Y, return_dist=True)
    u_arr, v_arr, k_arr = zip(*edges)
    carreaux[f"snap_u_{suf}"]    = list(u_arr)
    carreaux[f"snap_v_{suf}"]    = list(v_arr)
    carreaux[f"snap_key_{suf}"]  = list(k_arr)
    carreaux[f"snap_dist_{suf}"] = dist_arr   # mètres

print("Snap-to-edge calculé pour les deux réseaux.")
carreaux[["idcar_200m",
          "snap_u_pieton", "snap_v_pieton", "snap_dist_pieton",
          "snap_u_velo",   "snap_v_velo",   "snap_dist_velo"]].head()

Audit des distances de rattachement (médiane, p95) par réseau.
Ces distances mesurent le biais résiduel du snap-to-edge : longueur du
tronçon d'approche entre le point représentatif et son pied de perpendiculaire
sur l'arête. Elles seront converties en budget-temps en §4 ; l'audit ici sert
à détecter d'éventuels carreaux aberrants (point hors emprise OSM).

In [ ]:
VITESSE_PIED_MS = 5  * 1000 / 3600   # m/s
VITESSE_VELO_MS = 15 * 1000 / 3600   # m/s
VITESSES_MS = {"piéton": VITESSE_PIED_MS, "vélo": VITESSE_VELO_MS}

for nom, G in GRAPHES.items():
    suf   = SUFFIXE[nom]
    col   = f"snap_dist_{suf}"
    v_ms  = VITESSES_MS[nom]
    dists = carreaux[col]

    mediane = dists.median()
    p95     = dists.quantile(0.95)
    maxi    = dists.max()

    print(
        f"Réseau {nom:<7} | "
        f"médiane {mediane:5.1f} m ({mediane / v_ms:4.1f} s) | "
        f"p95 {p95:5.1f} m ({p95 / v_ms:4.1f} s) | "
        f"max {maxi:6.1f} m ({maxi / v_ms:5.1f} s)"
    )

# Carreaux à distance de snap élevée (> 200 m) : à inspecter sous QGIS
SEUIL_SNAP_CARREAUX = 200   # m
for nom in GRAPHES:
    suf = SUFFIXE[nom]
    suspects = carreaux[carreaux[f"snap_dist_{suf}"] > SEUIL_SNAP_CARREAUX]
    if len(suspects):
        print(
            f"  ⚠ {len(suspects)} carreau(x) à snap_dist_{suf} > {SEUIL_SNAP_CARREAUX} m "
            f"(à inspecter)"
        )
    else:
        print(f"  ✓ Aucun carreau à snap_dist_{suf} > {SEUIL_SNAP_CARREAUX} m")

In [ ]:
carreaux.drop(columns=["pt_repr"]).to_file(DATA_DIR / "carreaux_snap.gpkg", driver="GPKG")

Les 15 carreaux à distance de snap > 200 m visualisés avec QGIS correspondent à des habitations isolées desservies par des voies privées absentes d'OSM ; la voie publique la plus proche constitue le rattachement le plus proche disponible. 

## §4 — Affectation du niveau atteignable par carreau

### Principe

Le snap-to-edge (§3) a projeté chaque carreau sur l'arête la plus proche.
Chaque arête a deux nœuds extrémités `u` et `v`. La table `acces_noeuds` de
nb03 indique, pour chaque nœud, le temps minimal pour atteindre chaque niveau
de desserte depuis n'importe quel arrêt de ce niveau.

Le niveau atteignable depuis le carreau est donc le meilleur niveau accessible
depuis `u` **ou** `v`, sous un budget de temps résiduel :

```
budget_résiduel = SEUIL (900 s) − t_approche
```

où `t_approche = snap_dist / vitesse` (secondes) est le temps pour rejoindre
le nœud extrémité depuis le point représentatif du carreau le long de l'arête.

La lecture se fait par (nœud, mode, niveau) dans `acces_noeuds` :
- si `temps_acces_s[nœud, mode, niveau] ≤ budget_résiduel` → niveau atteignable
- on retient le niveau d'ordre maximal parmi ceux atteignables depuis `u` ou `v`
- si aucun niveau n'est atteignable → `niveau_ordre = 0` (« aucun »)

Le calcul est mené pour les deux horizons (2026, SERM) et les deux modes
(piéton, vélo), soit quatre combinaisons par carreau.

Construction des tables d'index pour la lecture rapide de `acces_noeuds`.
Un dictionnaire `(mode, node) → [(niveau_ordre, temps_acces_s)]` permet
d'éviter une jointure complète répétée pour chaque carreau.

In [ ]:
from collections import defaultdict

def construire_index(gdf_acces):
    """Construit un dict (mode_deplacement, node) → liste de (niveau_ordre, temps_acces_s)
    depuis la table acces_noeuds d'un horizon. Permet la lecture O(1) par nœud."""
    idx = defaultdict(list)
    for _, row in gdf_acces.iterrows():
        idx[(row["mode_deplacement"], row["node"])].append(
            (row["niveau_ordre"], row["temps_acces_s"])
        )
    return idx

index_acces = {h: construire_index(gdf) for h, gdf in acces_noeuds.items()}
print("Index construits :")
for h, idx in index_acces.items():
    print(f"  horizon {h} : {len(idx):,} clés (mode × nœud)")

Calcul du meilleur niveau atteignable par carreau × horizon × mode.

Pour chaque carreau, les nœuds `u` et `v` de l'arête snappée sont lus depuis
les colonnes `snap_u_{suf}` et `snap_v_{suf}`. Le budget résiduel est calculé
depuis les deux extrémités — la plus favorable est retenue. Le résultat est
stocké dans un DataFrame long (une ligne par carreau × horizon × mode), qui
sera pivoté au §5 en colonnes `niveau_actuel` et `niveau_cible`.

In [ ]:
MODES_DEPL = {"piéton": "pieton", "vélo": "velo"}
VITESSES_S = {"piéton": 5 * 1000 / 3600, "vélo": 15 * 1000 / 3600}  # m/s

def meilleur_niveau_depuis_noeud(node, mode, budget_s, idx):
    """Meilleur niveau_ordre atteignable depuis `node` sous `budget_s` secondes.
    Retourne 0 si aucun niveau n'est atteignable."""
    entrees = idx.get((mode, node), [])
    atteignables = [niv for niv, t in entrees if t <= budget_s]
    return max(atteignables, default=0)

lignes = []
for horizon, idx in index_acces.items():
    for mode_depl, suf in MODES_DEPL.items():
        v_ms = VITESSES_S[mode_depl]
        col_u    = f"snap_u_{suf}"
        col_v    = f"snap_v_{suf}"
        col_dist = f"snap_dist_{suf}"

        for _, row in carreaux.iterrows():
            dist_m  = row[col_dist]
            t_app_s = dist_m / v_ms          # temps d'approche en secondes

            # Budget résiduel depuis chaque extrémité de l'arête
            # (le carreau peut être plus proche de u que de v ou l'inverse ;
            # on prend le meilleur des deux sans hypothèse sur la position exacte)
            budget_u = SEUIL_ACCESSIBILITE_S - t_app_s
            budget_v = SEUIL_ACCESSIBILITE_S - t_app_s   # symétrique : même dist de snap

            niv_u = meilleur_niveau_depuis_noeud(row[col_u], mode_depl, budget_u, idx)
            niv_v = meilleur_niveau_depuis_noeud(row[col_v], mode_depl, budget_v, idx)
            niv   = max(niv_u, niv_v)

            lignes.append({
                "idcar_200m":   row["idcar_200m"],
                "horizon":      horizon,
                "mode":         mode_depl,
                "niveau_ordre": niv,
            })

resultats = pd.DataFrame(lignes)
print(f"Résultats : {len(resultats):,} lignes (carreaux × horizons × modes)")
print(resultats.groupby(["horizon", "mode", "niveau_ordre"]).size().to_string())

Jointure du libellé de niveau et du nom de tranche sur les résultats.
`niveau_ordre = 0` correspond à « aucun » — aucun niveau structurant atteignable
sous 15 min depuis ce carreau dans ce mode et cet horizon.

In [ ]:
LIBELLE_NIVEAU = {0: "aucun", 1: "Bus", 2: "TEOR", 3: "Métro", 4: "Car express", 5: "Train"}

resultats["niveau"] = resultats["niveau_ordre"].map(LIBELLE_NIVEAU)

# Contrôles
assert resultats["niveau_ordre"].between(0, 5).all(), "niveau_ordre hors [0, 5]"
assert resultats["niveau"].notna().all(), "niveau_ordre sans libellé"
assert len(resultats) == len(carreaux) * 2 * 2, (
    f"Comptage inattendu : {len(resultats)} lignes "
    f"(attendu {len(carreaux) * 2 * 2} = {len(carreaux)} carreaux × 2 horizons × 2 modes)"
)

print("Répartition des niveaux atteignables (toutes combinaisons) :")
print(
    resultats.groupby(["horizon", "mode", "niveau"])["idcar_200m"]
    .count()
    .rename("carreaux")
    .to_string()
)

Run §4 OK.Run §4 OK. Les résultats sont analytiquement cohérents et porteurs de sens.
Quelques lectures rapides avant le §5.
Horizon 2026. À pied, 338 carreaux sans accès structurant (« aucun ») ; à vélo, seulement 53 — le vélo réduit drastiquement les zones blanches. Le train est le niveau le plus atteint à vélo (1 996 carreaux) contre seulement 393 à pied : c'est la signature attendue des gares dispersées, dont l'isochrone piétonne est étroite mais l'isochrone cyclable très étendue.
Horizon SERM. Le car express apparaît bien (581 carreaux à pied, 1 020 à vélo) — preuve que la réintégration dans STRUCTURANTS fonctionne. Le train progresse fortement (393→984 à pied, 1 996→3 031 à vélo). Le métro recule légèrement à vélo (380→12) : des carreaux qui atteignaient le métro en 2026 atteignent désormais un niveau supérieur (car express ou train) à l'horizon SERM — ils sont correctement reclassés vers le haut. C'est exactement ce que l'overlay additif doit produire.
Un point à vérifier. À l'horizon SERM à vélo, le TEOR chute de 476 à 118 et le métro de 380 à 12. Ce recul du TEOR et du métro est cohérent si des carreaux montent en hiérarchie vers car express ou train — mais l'ampleur du recul métro vélo (−368) mérite une vérification rapide en §6 : est-ce que ces 368 carreaux se retrouvent bien en car express ou train SERM ? Ce sera naturel à contrôler lors de la validation croisée.

## §5 — Indicateur composite d'accessibilité par carreau

C'est ici que naît l'**accessibilité** au sens du cadrage, par opposition à
l'atteignabilité de nb03.

nb03 répondait à : *depuis cet arrêt, quels nœuds sont accessibles ?*
nb05 §4 a répondu à : *depuis ce carreau, quel niveau est atteignable ?*
nb05 §5 répond à : *combien d'habitants atteignent quel niveau, et le SERM
change-t-il ce portrait ?*

Le §4 a produit un DataFrame long `resultats` (23 660 lignes :
5 915 carreaux × 2 horizons × 2 modes). Ce §5 le pivote en colonnes
`niveau_actuel` / `niveau_cible` par carreau × mode, y joint la population
`ind_pond`, et calcule les agrégats qui formeront le livrable analytique.

### Pivot du tableau long en colonnes par horizon

Chaque carreau × mode reçoit deux colonnes de niveau : `niveau_actuel` (horizon
2026) et `niveau_cible` (horizon SERM). Le pivot est effectué sur `niveau_ordre`
(entier) — le libellé textuel est rejoint ensuite pour éviter de pivoter des
chaînes de caractères.

In [ ]:
pivot = (
    resultats[["idcar_200m", "horizon", "mode", "niveau_ordre"]]
    .pivot_table(
        index=["idcar_200m", "mode"],
        columns="horizon",
        values="niveau_ordre",
        aggfunc="max",     # un seul niveau par (carreau, mode, horizon) — max = identité
    )
    .reset_index()
    .rename(columns={"2026": "niveau_actuel", "SERM": "niveau_cible"})
)

# Libellés textuels
LIBELLE_NIVEAU = {0: "aucun", 1: "Bus", 2: "TEOR", 3: "Métro", 4: "Car express", 5: "Train"}
pivot["libelle_actuel"] = pivot["niveau_actuel"].map(LIBELLE_NIVEAU)
pivot["libelle_cible"]  = pivot["niveau_cible"].map(LIBELLE_NIVEAU)

# Gain de niveau (positif = amélioration, 0 = stable, négatif = impossible par construction)
pivot["gain_niveau"] = pivot["niveau_cible"] - pivot["niveau_actuel"]

assert (pivot["gain_niveau"] >= 0).all(),     "Gain négatif détecté — un carreau régresserait entre 2026 et SERM"

print(f"Pivot : {len(pivot):,} lignes (carreaux × modes)")
print(pivot.groupby("mode")["gain_niveau"].value_counts().sort_index().to_string())

### Jointure de la population

La population `ind_pond` du carreau est jointe depuis `carreaux`. Elle est
identique pour les deux modes d'un même carreau (c'est une propriété du carreau,
pas du mode de déplacement) — la jointure est donc faite sur `idcar_200m` seul.

In [ ]:
pop_carr = carreaux[["idcar_200m", "ind_pond"]].copy()

acces_pop = pivot.merge(pop_carr, on="idcar_200m", how="left")

assert acces_pop["ind_pond"].notna().all(),     "Carreaux sans population après jointure"
assert len(acces_pop) == len(pivot),     "Lignes perdues à la jointure population"

print(f"Population totale réconciliée : {acces_pop['ind_pond'].sum():,.0f}")
print(f"  (attendu : {2*POP_TOTALE:,.0f}  — deux lignes par carreau, une par mode)")

### Agrégats d'accessibilité pondérés par la population

Deux lectures complémentaires :

- **Par niveau atteint** : combien d'habitants atteignent au moins tel niveau,
  par mode et par horizon ? C'est l'indicateur d'équité territoriale du cadrage.
- **Gain SERM** : combien d'habitants gagnent au moins un niveau de desserte
  grâce au réseau cible ?

Les deux sont calculés en population pondérée (`ind_pond`), pas en nombre de
carreaux — un carreau de 200 m peut porter 0 à plusieurs centaines d'habitants.

In [ ]:
def pop_atteignant_niveau(df, mode, niveau_min, col_niveau):
    """Population (ind_pond) atteignant au moins `niveau_min` pour le mode donné."""
    masque = (df["mode"] == mode) & (df[col_niveau] >= niveau_min)
    return df.loc[masque, "ind_pond"].sum()

def pop_exactement_niveau(df, mode, niveau_exact, col_niveau):
    """Population (ind_pond) atteignant exactement `niveau_exact` pour le mode donné."""
    masque = (df["mode"] == mode) & (df[col_niveau] == niveau_exact)
    return df.loc[masque, "ind_pond"].sum()

SEP = "-" * 66
HDR = f"{'Niveau':<14} {'Pied 2026':>12} {'Pied SERM':>12} {'Vélo 2026':>12} {'Vélo SERM':>12}"

print("=== Population atteignant chaque niveau (au moins) ===")
print(HDR); print(SEP)
for niv_ord, niv_lbl in sorted(LIBELLE_NIVEAU.items()):
    if niv_ord == 0:
        continue
    row = [niv_lbl]
    for mode in ("piéton", "vélo"):
        for col in ("niveau_actuel", "niveau_cible"):
            pop = pop_atteignant_niveau(acces_pop, mode, niv_ord, col)
            row.append(f"{pop:>12,.0f}")
    print(f"{row[0]:<14} {row[1]} {row[2]} {row[3]} {row[4]}")

print()
print("=== Population atteignant chaque niveau (exactement) ===")
print(HDR); print(SEP)
for niv_ord, niv_lbl in sorted(LIBELLE_NIVEAU.items()):
    row = [niv_lbl]
    for mode in ("piéton", "vélo"):
        for col in ("niveau_actuel", "niveau_cible"):
            pop = pop_exactement_niveau(acces_pop, mode, niv_ord, col)
            row.append(f"{pop:>12,.0f}")
    print(f"{row[0]:<14} {row[1]} {row[2]} {row[3]} {row[4]}")

print()
print("=== Gain SERM : habitants gagnant au moins un niveau de desserte ===")
for mode in ("piéton", "vélo"):
    gagnants = acces_pop.loc[
        (acces_pop["mode"] == mode) & (acces_pop["gain_niveau"] >= 1),
        "ind_pond"
    ].sum()
    pct = gagnants / POP_TOTALE * 100
    print(f"  {mode:<7} : {gagnants:>10,.0f} hab. ({pct:.1f} % de la population MRN)")

### Constitution de la couche géographique par carreau

Le DataFrame `acces_pop` est encore tabulaire. Pour l'export en GeoPackage et
la restitution cartographique, on y joint la géométrie des carreaux (polygones
200 m). La couche résultante `carreaux_accessibilite` est la base du §7 et des
cartes de restitution QGIS.

In [ ]:
carreaux_geo = carreaux[["idcar_200m", "geometry"]].copy()

carreaux_accessibilite = carreaux_geo.merge(acces_pop, on="idcar_200m", how="left")
carreaux_accessibilite = gpd.GeoDataFrame(
    carreaux_accessibilite, geometry="geometry", crs=CRS_PROJET
)

assert len(carreaux_accessibilite) == len(acces_pop),     "Lignes perdues à la jointure géométrique"
assert carreaux_accessibilite.geometry.notna().all(),     "Géométrie(s) nulle(s) après jointure"

print(f"Couche géographique : {len(carreaux_accessibilite):,} entités")
print(f"Colonnes : {list(carreaux_accessibilite.columns)}")

## §6 — Validation croisée

Deux vérifications indépendantes de la cohérence des résultats du §4-§5 :

1. **Point-dans-polygone contre `isochrones_2026`** : approche alternative au
   snap-to-edge. Pour chaque carreau, on vérifie si son point représentatif tombe
   dans une isochrone 2026. Le niveau lu par cette méthode doit coïncider avec
   `niveau_actuel` pour la grande majorité des carreaux — les divergences
   résiduelles s'expliquent par la différence de méthode (isochrones dissoutes
   avec tampon cartographique vs. routage exact nœud par nœud).

2. **Réconciliation des totaux de population** : la somme de `ind_pond` sur les
   carreaux doit rester stable entre le chargement (§2) et la couche finale (§5).
   Toute dérive signalerait une perte de carreaux à une jointure.

### 6.1 — Point-dans-polygone contre les isochrones 2026

Les isochrones de nb03 sont des polygones dissous par (mode, niveau, tranche).
Pour chaque carreau, on cherche le meilleur niveau dont l'isochrone à 15 min
contient le point représentatif — c'est la méthode alternative la plus directe.

Note méthodologique : les isochrones portent un tampon cartographique
(`TAMPON_ISO = 50 m`) et sont dissoutes sur l'union des arrêts d'un même niveau.
Elles peuvent donc couvrir des zones légèrement plus larges que le routage exact.
Les divergences attendues sont donc des carreaux classés à un niveau supérieur
par le point-dans-polygone — jamais à un niveau inférieur.

In [ ]:
iso_2026 = gpd.read_file(DATA_DIR / "isochrones_2026.gpkg")

# Conserver uniquement la tranche 15 min (budget complet) par mode et niveau
iso_15 = iso_2026[iso_2026["tranche_min"] == 15].copy()

# Point représentatif des carreaux (déjà calculé en §3, recalculé ici pour
# ne pas dépendre de l'ordre d'exécution)
pts = carreaux[["idcar_200m"]].copy()
pts["geometry"] = carreaux.geometry.representative_point()
pts = gpd.GeoDataFrame(pts, geometry="geometry", crs=CRS_PROJET)

# Jointure spatiale : chaque point reçoit tous les polygones isochrones qui le
# contiennent, on retient le niveau_ordre maximal par (carreau, mode)
pip = gpd.sjoin(pts, iso_15[["mode_deplacement", "niveau_ordre", "geometry"]],
                how="left", predicate="within")

pip_max = (
    pip.groupby(["idcar_200m", "mode_deplacement"])["niveau_ordre"]
    .max()
    .reset_index()
    .rename(columns={"niveau_ordre": "niveau_pip", "mode_deplacement": "mode"})
)
pip_max["niveau_pip"] = pip_max["niveau_pip"].fillna(0).astype(int)

print(f"Point-dans-polygone calculé : {len(pip_max):,} lignes (carreaux × modes)")
print(pip_max.groupby("mode")["niveau_pip"].value_counts().sort_index().to_string())

Comparaison avec `niveau_actuel` (résultat du §4). On mesure le taux de
concordance et on inspecte les divergences.

In [ ]:
# Jointure avec les résultats §4 (horizon 2026 uniquement)
actuel_2026 = acces_pop[acces_pop["mode"].isin(["piéton", "vélo"])][
    ["idcar_200m", "mode", "niveau_actuel"]
].copy()

comparaison = actuel_2026.merge(pip_max, on=["idcar_200m", "mode"], how="left")
comparaison["niveau_pip"] = comparaison["niveau_pip"].fillna(0).astype(int)
comparaison["concordant"] = comparaison["niveau_actuel"] == comparaison["niveau_pip"]

n_total      = len(comparaison)
n_concordant = comparaison["concordant"].sum()
print(f"Concordance totale : {n_concordant:,} / {n_total:,} "
      f"({n_concordant / n_total:.1%})")
print()

divergences = comparaison[~comparaison["concordant"]].copy()
divergences["sens"] = (divergences["niveau_pip"] - divergences["niveau_actuel"]).apply(
    lambda d: "pip > actuel (tampon iso)" if d > 0 else "pip < actuel"
)
print("Divergences par sens et mode :")
print(divergences.groupby(["mode", "sens"]).size().to_string())
print()

# Divergences pip < actuel : attendues pour les carreaux dont le point
# représentatif est à plus de 50 m de toute arête atteinte (TAMPON_ISO = 50 m).
# Le snap-to-edge projette sur l'arête et lit directement la table nœud ;
# le pip échoue dès que le point tombe dans l'interstice entre arêtes.
# Ces divergences confirment que le snap-to-edge est la bonne méthode —
# plus précis que le pip précisément là où le pip est en défaut.
pip_inferieur = divergences[divergences["sens"] == "pip < actuel"]
if len(pip_inferieur):
    cas_bus    = pip_inferieur[pip_inferieur["niveau_actuel"] == 1]
    cas_struct = pip_inferieur[pip_inferieur["niveau_actuel"] > 1]
    print(f"Divergences pip < actuel : {len(pip_inferieur)} au total")
    print(f"  Bus → aucun (point représentatif > 50 m d'une arête atteinte) : {len(cas_bus)}")
    print(f"  Niveau structurant → inférieur (à noter)                       : {len(cas_struct)}")
    if len(cas_struct):
        print(cas_struct[["idcar_200m", "mode", "niveau_actuel", "niveau_pip"]])
    else:
        print("  ✓ Aucun cas structurant inattendu.")
else:
    print("✓ Aucune divergence pip < actuel.")

### 6.2 — Réconciliation des totaux de population

La somme de `ind_pond` doit rester identique entre le §2 (chargement) et la
couche finale du §5, pour les deux modes séparément et pour la somme totale.

In [ ]:
# Population dans la couche finale, par mode
pop_finale = (
    carreaux_accessibilite
    .groupby("mode")["ind_pond"]
    .sum()
)
print("Population dans la couche finale par mode :")
print(pop_finale.to_string())
print(f"Total (× 2 modes) : {pop_finale.sum():,.0f}")
print(f"Référence §2      : {POP_TOTALE:,.0f} × 2 = {POP_TOTALE * 2:,.0f}")
print()

for mode in ("piéton", "vélo"):
    ecart = abs(pop_finale[mode] - POP_TOTALE)
    assert ecart < 1, (
        f"Population {mode} diverge de §2 : {pop_finale[mode]:,.0f} ≠ {POP_TOTALE:,.0f}"
    )
print("✓ Totaux de population réconciliés — aucune perte de carreaux.")

### 6.3 — Export de contrôle pour QGIS

Export d'une couche légère contenant les colonnes de validation :
point représentatif, `niveau_actuel`, `niveau_pip`, flag `concordant`.
Permet une inspection visuelle des divergences sous QGIS sans charger
la couche complète.

In [ ]:
controle = comparaison.merge(
    carreaux[["idcar_200m", "geometry"]], on="idcar_200m", how="left"
)
controle = gpd.GeoDataFrame(controle, geometry="geometry", crs=CRS_PROJET)

controle.to_file(DATA_DIR / "controle_validation_2026.gpkg", driver="GPKG")
print(f"✓ Exporté : controle_validation_2026.gpkg ({len(controle):,} entités)")
print(f"  Colonnes : {list(controle.columns)}")

## §7 — Export et limites

Export du livrable analytique principal et documentation des limites
spécifiques à nb05, à intégrer dans le README.

### 7.1 — Export de `population_accessibilite_MRN.gpkg`

Le GeoPackage exporté contient une entité par carreau × mode de déplacement.
Les colonnes analytiques principales sont :

| Colonne | Description |
|---|---|
| `idcar_200m` | Identifiant carreau INSEE 200 m |
| `mode` | Mode de déplacement (`"piéton"` \| `"vélo"`) |
| `niveau_actuel` | Meilleur niveau atteignable, horizon 2026 (0 = aucun … 5 = Train) |
| `niveau_cible` | Meilleur niveau atteignable, réseau cible SERM |
| `libelle_actuel` | Libellé textuel de `niveau_actuel` |
| `libelle_cible` | Libellé textuel de `niveau_cible` |
| `gain_niveau` | Gain de niveau SERM − 2026 (≥ 0 par construction) |
| `ind_pond` | Population Filosofi pondérée du carreau |
| `geometry` | Polygone carreau 200 m, EPSG:2154 |

Les colonnes de snap (`snap_u_*`, `snap_v_*`, `snap_key_*`, `snap_dist_*`)
et le point représentatif (`pt_repr`) sont exclus de l'export — données
intermédiaires sans valeur analytique pour les utilisateurs en aval.

In [ ]:
COLS_EXPORT = [
    "idcar_200m", "mode",
    "niveau_actuel", "libelle_actuel",
    "niveau_cible",  "libelle_cible",
    "gain_niveau", "ind_pond", "geometry",
]

sortie = DATA_DIR / "population_accessibilite_MRN.gpkg"

(
    carreaux_accessibilite[COLS_EXPORT]
    .to_file(sortie, driver="GPKG", layer="population_accessibilite")
)

print(f"✓ Exporté : {sortie.name}")
print(f"  {len(carreaux_accessibilite):,} entités | CRS : {carreaux_accessibilite.crs}")
print(f"  Colonnes exportées : {COLS_EXPORT}")

### 7.2 — Limites spécifiques à nb05

Ces limites complètent les entrées générales du README (écart de population
Filosofi, précision du carroyage, horizon de comparaison).

---

**Rattachement des carreaux au réseau (snap-to-edge).**
Chaque carreau est rattaché par son point représentatif à l'arête OSM la plus
proche. La distance de snap (médiane ~22 m, p95 ~111 m, max 344 m) constitue
un biais résiduel de localisation, du même ordre que l'imprécision du carroyage
200 m (~0 à 100 m). Les 15 carreaux à distance > 200 m correspondent à des
habitations isolées desservies par des voies privées absentes d'OSM ; la voie
publique la plus proche constitue le rattachement disponible le plus proche.

**Correction du tronçon d'approche.**
Le temps de parcours du point représentatif jusqu'au nœud extrémité de l'arête
est estimé depuis la distance orthogonale de snap (distance minimale
point → arête), qui est une borne inférieure de la distance curviligne réelle
jusqu'à `u` ou `v`. La correction est donc légèrement optimiste — elle
sous-estime marginalement le tronçon d'approche. Cette approximation est
cohérente avec la précision du carroyage.

**Densité uniforme sur les carreaux de bordure.**
Les carreaux tronqués par la limite de la MRN voient leur population pondérée
au prorata de la surface conservée (`ind_pond = ind × frac`), sous hypothèse
de densité uniforme à l'intérieur du carreau. Cette hypothèse introduit une
erreur systématique pour les carreaux dont la population est concentrée dans
la partie exclue (ex. : bourg en limite de MRN).

**Colonnes parasites dans `population_carreaux_MRN.gpkg`.**
Le fichier produit par nb04 contient des colonnes issues du géocodage OSMnx
(`bbox_west`, `place_id`, `display_name`, etc.), sans incidence analytique
sur nb05 mais à supprimer dans une prochaine version de nb04.

**Validation croisée par point-dans-polygone.**
La concordance avec les isochrones 2026 est de 74,6 %. Les 25,4 % de
divergences sont quasi-exclusivement des carreaux dont le point représentatif
est à plus de 50 m de toute arête atteinte (TAMPON_ISO = 50 m étant un
paramètre cartographique, non analytique) — classés Bus par snap-to-edge,
non couverts par l'isochrone dissoute. Ces divergences confirment la
supériorité du snap-to-edge sur le point-dans-polygone pour les carreaux
éloignés des arêtes, et ne constituent pas des anomalies.

In [ ]:
# Bilan synthétique pour le README / publication
print("=== Bilan nb05 ===")
print(f"Carreaux traités          : {len(carreaux):,}")
print(f"Population totale (Filosofi) : {POP_TOTALE:,.0f} hab.")
print()
print("Gain SERM — habitants gagnant au moins un niveau de desserte :")
for mode in ("piéton", "vélo"):
    gagnants = acces_pop.loc[
        (acces_pop["mode"] == mode) & (acces_pop["gain_niveau"] >= 1),
        "ind_pond"
    ].sum()
    pct = gagnants / POP_TOTALE * 100
    print(f"  {mode:<7} : {gagnants:>10,.0f} hab. ({pct:.1f} %)")
print()
print("Sortie : population_accessibilite_MRN.gpkg")
print("Étape suivante : nb06 — comparaison avant / après réseau cible SERM")